<a href="https://colab.research.google.com/github/jgham0101/yolo-edge-optimization/blob/main/Week1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print(torch.cuda.is_available())

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

In [ ]:
model.train(
    data="coco128.yaml",
    epochs=20,
    imgsz=640
)

In [ ]:
metrics = model.val()
print(metrics)

In [ ]:
model.predict(source="https://ultralytics.com/images/bus.jpg", save=True)

In [ ]:
# 1. mAP 측정

metrics = model.val()

# mAP 값 추출
map50 = metrics.box.map50
map5095 = metrics.box.map

print("mAP50:", map50)
print("mAP50-95:", map5095)


# 2. FPS 측정

import time

# 테스트 이미지
test_image = "https://ultralytics.com/images/bus.jpg"

# 평균 내기
runs = 10
total_time = 0

for _ in range(runs):
    start = time.time()
    model.predict(source=test_image, verbose=False)
    end = time.time()
    total_time += (end - start)

avg_time = total_time / runs
fps = 1 / avg_time

print("Average FPS:", fps)


# 3. Model Size 측정

import os

model_path = "runs/detect/train/weights/best.pt"

size_mb = os.path.getsize(model_path) / (1024 * 1024)

print("Model Size (MB):", size_mb)


# 4. 결과 표 생성

import pandas as pd

results = pd.DataFrame({
    "Model": ["Baseline (YOLOv8n)"],
    "mAP50": [round(map50, 4)],
    "mAP50-95": [round(map5095, 4)],
    "FPS": [round(fps, 2)],
    "Model Size (MB)": [round(size_mb, 2)]
})

print("\n=== Final Results ===")
print(results)

In [ ]:
import matplotlib.pyplot as plt

labels = ["mAP50", "FPS", "Model Size"]
values = [map50, fps, size_mb]

plt.figure()
plt.bar(labels, values)
plt.title("Baseline Model Performance")
plt.xlabel("Metrics")
plt.ylabel("Values")
plt.show()